# ATLAS hydro CMIP6 delta scenarios

This notebook applies CMIP6 climate change signals to the present-day river discharge climatology derived from GLOFAS.

The workflow starts by loading the preprocessed CMIP6 historical and future scenario datasets together with the GLOFAS river discharge climatology, previously integrated with in-situ hydrological station observations and aggregated over HydroBASINS catchments.

Since CMIP6 river discharge projections have a spatial resolution of approximately 100 km, while GLOFAS data are available at about 5.5 km resolution, the CMIP6 datasets are first carefully interpolated onto the GLOFAS grid. Because river discharge is a highly non-linear variable and interpolation may introduce artefacts outside the river network, the interpolated fields are subsequently clipped using the HydroRIVERS dataset, reducing spurious values and preserving the spatial structure of the river system.

Monthly climate change signals are then calculated by comparing each future SSP experiment against the corresponding CMIP6 historical experiment. The resulting monthly percentage change represents the projected variation relative to the historical climatology.

These percentage changes are subsequently applied to the station-integrated GLOFAS climatology at HydroBASINS level 12, generating future river discharge estimates that preserve the spatial detail and local calibration of the observational baseline while incorporating the climate change signal from CMIP6 projections.

Finally, both the projected river discharge and the corresponding percentage change are aggregated from HydroBASINS level 12 to HydroBASINS level 5 and saved for subsequent visualization and analysis.

## Step 1. Import libraries

This step loads the Python libraries used to read NetCDF files, manage geospatial layers, compute zonal statistics and save the final outputs.

In [1]:
from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # noqa: F401. Required to use the .rio accessor on xarray objects.
import rasterio as rio
import rasterstats as rstats

warnings.filterwarnings("ignore")

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


## Step 2. Define user parameters

Edit only this cell to change country, model, scenario, period or folders.

The CMIP6 inputs must come from the CMIP6 preprocessing notebook. The expected structure is:

`../data/processed/river_discharge/{COUNTRY_FOLDER}/{experiment}/{MODEL}/`

The integrated GLOFAS basin statistics must come from the basin statistics and station integration workflow. The expected structure is:

`../data/basin_statistics/{COUNTRY_FOLDER}/`

In [15]:
# ---------------------------------------------------------------------
# Main configuration
# ---------------------------------------------------------------------

COUNTRY_FOLDER = "ecuador"
COUNTRY_NAME_IN_SHAPEFILE = "Ecuador"

# CMIP6 variable name. For river discharge, use 'rivo'.
VARIABLE = "rivo"
VARIABLE_LONG_NAME = "river_discharge"

MODEL = "CNRM-ESM2-1"
HISTORICAL_EXPERIMENT = "historical"
SCENARIO_EXPERIMENT = "ssp370"

# Scenario period used to compute the future monthly climatology.
SCENARIO_START = "2020"
SCENARIO_END = "2050"

# Periods used in the filenames produced by the preprocessing notebooks.
GLOFAS_START = "1991-01"
GLOFAS_END = "1991-01"
HISTORICAL_START = "1985-01"
HISTORICAL_END = "2014-12"
SCENARIO_FILE_START = "2015-01"
SCENARIO_FILE_END = "2100-12"

# Grid cells with present GLOFAS discharge <= this value are ignored.
RIVER_DISCHARGE_THRESHOLD = 1

# ---------------------------------------------------------------------
# Folders and files
# ---------------------------------------------------------------------

GLOFAS_FILE = (
    Path("../data/processed")
    / VARIABLE_LONG_NAME
    / COUNTRY_FOLDER
    / f"river_discharge_{GLOFAS_START}_{GLOFAS_END}_processed.nc"
)

CMIP6_PROCESSED_DIR = Path("../data/processed") / VARIABLE_LONG_NAME / COUNTRY_FOLDER

HISTORICAL_FILE = (
    CMIP6_PROCESSED_DIR
    / MODEL
    / HISTORICAL_EXPERIMENT

    / f"{VARIABLE}_{HISTORICAL_START}_{HISTORICAL_END}_processed.nc"
)

SCENARIO_FILE = (
    CMIP6_PROCESSED_DIR
    / MODEL
    / SCENARIO_EXPERIMENT

    / f"{VARIABLE}_{SCENARIO_FILE_START}_{SCENARIO_FILE_END}_processed.nc"
)

# Integrated present-day basin statistics.
# These CSV files must contain a geometry column in WKT format.
BASIN_STATISTICS_DIR = Path("../data/atlas_data") / COUNTRY_FOLDER

OUTPUT_DIR = (
    Path("../data/atlas_data")
    / COUNTRY_FOLDER
    / MODEL
    / SCENARIO_EXPERIMENT
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Static geospatial inputs.
SHAPEFILE_PATH = Path("../world_map/ne_50m_admin_0_countries.shp")
HYDRO_RIVERS_PATH = Path("../hydrorivers/HydroRIVERS_v10_sa_shp/HydroRIVERS_v10_sa.shp")
HYDROBASINS_DIR = Path("../hydrobasins")

print("Country folder:", COUNTRY_FOLDER)
print("Country name in shapefile:", COUNTRY_NAME_IN_SHAPEFILE)
print("GLOFAS file:", GLOFAS_FILE)
print("Historical CMIP6 file:", HISTORICAL_FILE)
print("Scenario CMIP6 file:", SCENARIO_FILE)
print("Input basin statistics folder:", BASIN_STATISTICS_DIR)
print("Output folder:", OUTPUT_DIR)

Country folder: ecuador
Country name in shapefile: Ecuador
GLOFAS file: ../data/processed/river_discharge/ecuador/river_discharge_1991-01_1991-01_processed.nc
Historical CMIP6 file: ../data/processed/river_discharge/ecuador/CNRM-ESM2-1/historical/rivo_1985-01_2014-12_processed.nc
Scenario CMIP6 file: ../data/processed/river_discharge/ecuador/CNRM-ESM2-1/ssp370/rivo_2015-01_2100-12_processed.nc
Input basin statistics folder: ../data/atlas_data/ecuador
Output folder: ../data/atlas_data/ecuador/CNRM-ESM2-1/ssp370


## Step 3. Define helper functions

These functions keep the workflow shorter and easier to read. They load geospatial inputs, prepare NetCDF files, compute interpolation and masks, and save the final basin statistics.

In [8]:
def read_country_geometry(shapefile_path, country_name):
    """Read the selected country geometry from the Natural Earth shapefile."""
    countries = gpd.read_file(shapefile_path)[["ADMIN", "geometry"]]
    countries = countries.set_index("ADMIN")
    if countries.crs is None:
        countries = countries.set_crs("EPSG:4326")
    countries = countries.to_crs("EPSG:4326")
    if country_name not in countries.index:
        raise ValueError(f"Country '{country_name}' was not found in {shapefile_path}")
    return countries.loc[[country_name]]


def retrieve_hydrobasins(level, country_geometry):
    """Load HydroBASINS for one level and clip them to the selected country."""
    level_string = str(level).zfill(2)
    basins_file = HYDROBASINS_DIR / f"hybas_lake_sa_lev{level_string}_v1c.shp"
    basins = gpd.read_file(basins_file).to_crs("EPSG:4326")
    lakes = basins[basins["LAKE"] == 1].copy()
    basins = basins[basins["LAKE"] == 0].copy()
    country_basins = gpd.clip(basins, country_geometry.geometry)
    country_lakes = gpd.clip(lakes, country_geometry.geometry)
    return country_basins, country_lakes


def retrieve_country_rivers(rivers_path, country_geometry):
    """Load HydroRIVERS and keep only rivers inside the selected country."""
    rivers = gpd.read_file(rivers_path).to_crs("EPSG:4326")
    return gpd.clip(rivers, country_geometry.geometry)


def prepare_dataset_for_spatial_operations(ds):
    """Ensure that an xarray dataset has spatial dimensions and EPSG:4326 CRS."""
    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    if ds.rio.crs is None:
        ds = ds.rio.write_crs("EPSG:4326", inplace=False)
    return ds


def get_monthly_climatology(ds, variable_name, start=None, end=None):
    """Select an optional time period and compute monthly climatological means."""
    if start is not None or end is not None:
        ds = ds.sel(time=slice(start, end))
    return ds[variable_name].groupby("time.month").mean(skipna=True)


def interpolate_to_glofas_grid(cmip_monthly_da, glofas_monthly_da, method="nearest"):
    """Interpolate CMIP6 monthly data to the GLOFAS grid."""
    return cmip_monthly_da.interp(
        latitude=glofas_monthly_da.latitude.values,
        longitude=glofas_monthly_da.longitude.values,
        method=method,
    )


def clip_dataarray_to_rivers(da, rivers_gdf, variable_name):
    """Clip a DataArray using the selected river geometries."""
    ds = da.to_dataset(name=variable_name)
    ds = prepare_dataset_for_spatial_operations(ds)
    clipped = ds.rio.clip(rivers_gdf.geometry.values, rivers_gdf.crs, all_touched=True)
    return clipped[variable_name]


def mask_cmip_using_glofas(cmip_da, glofas_da, threshold):
    """Keep CMIP6 values only where GLOFAS discharge is above the selected threshold."""
    glofas_aligned, cmip_aligned = xr.align(glofas_da, cmip_da, join="right")
    return cmip_aligned.where(glofas_aligned > threshold)


def compute_percent_change(historical_da, scenario_da):
    """Compute monthly percentage change between scenario and historical CMIP6."""
    historical_aligned, scenario_aligned = xr.align(historical_da, scenario_da)
    delta = scenario_aligned - historical_aligned
    percent_change = xr.where(
        historical_aligned != 0,
        (delta / historical_aligned) * 100,
        np.nan,
    )
    percent_change.name = "rivo_percent_change"
    return percent_change.to_dataset()


def read_integrated_l12_basin_statistics(month):
    """Read one monthly integrated level 12 basin statistics CSV."""
    input_file = BASIN_STATISTICS_DIR / f"river_discharge_l12_{COUNTRY_FOLDER}_m{month}_integrated.csv"
    if not input_file.exists():
        raise FileNotFoundError(f"Missing input file: {input_file}")
    df = pd.read_csv(input_file)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns="Unnamed: 0")
    if "geometry" not in df.columns:
        raise ValueError(
            f"The file {input_file} does not contain a 'geometry' column. "
            "Run the corrected basin statistics notebook first, saving geometry as WKT."
        )
    df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"])
    return gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")


def get_affine_from_reference_file(reference_file):
    """Read the raster affine transform from the reference GLOFAS NetCDF file."""
    with rio.open(reference_file) as src:
        return src.transform


def aggregate_percent_change_on_basins(percent_change_ds, country_basins, month, affine):
    """Aggregate monthly percentage change over HydroBASINS polygons."""
    gdf = country_basins.copy()
    xdf_month = percent_change_ds.sel(month=month).drop_vars("month")
    variable_name = list(xdf_month.data_vars)[0]
    mean_basins = rstats.zonal_stats(
        gdf.geometry,
        xdf_month[variable_name].values,
        affine=affine,
        stats="mean",
    )
    gdf["mean_percent_change"] = [stat["mean"] for stat in mean_basins]
    gdf["mean_percent_change"] = gdf["mean_percent_change"].fillna(0)
    return gdf


def apply_percent_delta_to_reanalysis(percent_change_l12, present_integrated_l12):
    """Apply CMIP6 percentage change to present integrated GLOFAS basin discharge."""
    delta = percent_change_l12[["HYBAS_ID", "mean_percent_change"]].copy()
    gdf_future = present_integrated_l12.merge(delta, on="HYBAS_ID", how="left")
    gdf_future["mean_percent_change"] = gdf_future["mean_percent_change"].fillna(0)
    gdf_future["mean_river_discharge_corrected"] = (
        gdf_future["mean_river_discharge_integrated"]
        * (1 + gdf_future["mean_percent_change"] / 100)
    )
    gdf_future["delta_river_discharge"] = (
        gdf_future["mean_river_discharge_corrected"]
        - gdf_future["mean_river_discharge_integrated"]
    )
    return gdf_future.drop(columns=["mean_river_discharge_integrated", "mean_percent_change"])


def aggregate_l12_to_l5(gdf_l12, country_basins_l5):
    """Aggregate corrected level 12 values to level 5 using SUB_AREA as weight."""
    gdf_l12 = gdf_l12.to_crs(country_basins_l5.crs)
    gdf_points = gdf_l12.copy()
    gdf_points["geometry"] = gdf_points.geometry.representative_point()
    joined = gpd.sjoin(
        gdf_points,
        country_basins_l5[["HYBAS_ID", "geometry"]],
        how="left",
        predicate="within",
    )
    joined["weighted_q"] = joined["mean_river_discharge_corrected"] * joined["SUB_AREA"]
    joined["weighted_delta_q"] = joined["delta_river_discharge"] * joined["SUB_AREA"]
    aggregated = (
        joined
        .groupby("HYBAS_ID_right")
        .agg(
            weighted_q=("weighted_q", "sum"),
            weighted_delta_q=("weighted_delta_q", "sum"),
            area=("SUB_AREA", "sum"),
        )
        .reset_index()
    )
    aggregated["mean_river_discharge_corrected"] = aggregated["weighted_q"] / aggregated["area"]
    aggregated["delta_river_discharge"] = aggregated["weighted_delta_q"] / aggregated["area"]
    aggregated = aggregated.rename(columns={"HYBAS_ID_right": "HYBAS_ID"})
    return country_basins_l5.merge(
        aggregated[["HYBAS_ID", "mean_river_discharge_corrected", "delta_river_discharge"]],
        on="HYBAS_ID",
        how="left",
    )


def save_geodataframe_to_csv(gdf, output_file):
    """Save a GeoDataFrame to CSV with geometry in WKT format."""
    output_file.parent.mkdir(parents=True, exist_ok=True)
    df = gdf.copy()
    df["geometry"] = df.geometry.to_wkt()
    df.to_csv(output_file, index=False)


def save_geodataframe_to_geojson(gdf, output_file):
    """Save a GeoDataFrame to GeoJSON."""
    output_file.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_file, driver="GeoJSON")

## Step 4. Load country, rivers and basins

This step loads the static geospatial layers and clips them to the selected country. Only river segments with `ORD_FLOW < 7` are used for the river mask.

In [9]:
country_geometry = read_country_geometry(SHAPEFILE_PATH, COUNTRY_NAME_IN_SHAPEFILE)
country_rivers = retrieve_country_rivers(HYDRO_RIVERS_PATH, country_geometry)
rivers_only = country_rivers[country_rivers["ORD_FLOW"] < 7].copy()

country_basins_l5, country_lakes_l5 = retrieve_hydrobasins(5, country_geometry)
country_basins_l12, country_lakes_l12 = retrieve_hydrobasins(12, country_geometry)
country_basins_l12["L5basin"] = [int(str(pfaf_id)[:5]) for pfaf_id in country_basins_l12["PFAF_ID"]]

print("Country boundary:", country_geometry.total_bounds)
print("Number of river segments used as mask:", len(rivers_only))
print("Number of level 5 basins:", len(country_basins_l5))
print("Number of level 12 basins:", len(country_basins_l12))

Country boundary: [-91.65415039  -4.990625   -75.24960937   1.45537109]
Number of river segments used as mask: 15287
Number of level 5 basins: 11
Number of level 12 basins: 2110


## Step 5. Load GLOFAS and CMIP6 datasets

This step opens the processed NetCDF files and computes monthly climatologies. The scenario file is cut to `SCENARIO_START` and `SCENARIO_END` before the monthly climatology is computed.

In [10]:
glofas_ds = xr.open_dataset(GLOFAS_FILE)
glofas_ds = prepare_dataset_for_spatial_operations(glofas_ds)
glofas_monthly = get_monthly_climatology(glofas_ds, variable_name="dis24")

historical_ds = xr.open_dataset(HISTORICAL_FILE)
historical_monthly = get_monthly_climatology(historical_ds, variable_name=VARIABLE)

scenario_ds = xr.open_dataset(SCENARIO_FILE)
scenario_monthly = get_monthly_climatology(
    scenario_ds,
    variable_name=VARIABLE,
    start=SCENARIO_START,
    end=SCENARIO_END,
)

print("GLOFAS monthly climatology:")
print(glofas_monthly)
print("\nHistorical CMIP6 monthly climatology:")
print(historical_monthly)
print("\nScenario CMIP6 monthly climatology:")
print(scenario_monthly)

GLOFAS monthly climatology:
<xarray.DataArray 'dis24' (month: 1, latitude: 130, longitude: 330)> Size: 172kB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]], dtype=float32)
Coordinates:
  * latitude     (latitude) float32 520B 1.475 1.425 1.375 ... -4.925 -4.975
  * longitude    (longitude) float32 1kB -91.67 -91.62 -91.58 ... -75.27 -75.23
    spatial_ref  int64 8B 0
  * month        (month) int64 8B 1
Attributes: (12/30)
    GRIB_paramId:                             240024
    GRIB_dataType:                            sfo
    GRIB_numberOfPoints:                      48672
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            avg
    ...                       

## Step 6. Interpolate CMIP6 to the GLOFAS grid

CMIP6 river discharge usually has a coarser grid than GLOFAS. Nearest-neighbour interpolation is used here because river discharge is not a smooth field like temperature.

In [11]:
historical_monthly_interp = interpolate_to_glofas_grid(
    cmip_monthly_da=historical_monthly,
    glofas_monthly_da=glofas_monthly,
    method="nearest",
)

scenario_monthly_interp = interpolate_to_glofas_grid(
    cmip_monthly_da=scenario_monthly,
    glofas_monthly_da=glofas_monthly,
    method="nearest",
)

print("Interpolated historical CMIP6:")
print(historical_monthly_interp)
print("\nInterpolated scenario CMIP6:")
print(scenario_monthly_interp)

Interpolated historical CMIP6:
<xarray.DataArray 'rivo' (month: 12, latitude: 130, longitude: 330)> Size: 2MB
array([[[        nan,         nan,         nan, ...,   41.341976,
           41.341976,   41.341976],
        [        nan,         nan,         nan, ...,   41.341976,
           41.341976,   41.341976],
        [        nan,         nan,         nan, ...,   41.341976,
           41.341976,   41.341976],
        ...,
        [        nan,         nan,         nan, ..., 9143.121   ,
         9143.121   , 9143.121   ],
        [        nan,         nan,         nan, ..., 9143.121   ,
         9143.121   , 9143.121   ],
        [        nan,         nan,         nan, ..., 9143.121   ,
         9143.121   , 9143.121   ]],

       [[        nan,         nan,         nan, ...,   26.842888,
           26.842888,   26.842888],
        [        nan,         nan,         nan, ...,   26.842888,
           26.842888,   26.842888],
        [        nan,         nan,         nan, ...,   26.8

## Step 7. Mask CMIP6 using the present river network

This step clips the GLOFAS monthly climatology to the river network, then keeps CMIP6 values only where present GLOFAS discharge is above the selected threshold.

In [12]:
glofas_monthly_masked = clip_dataarray_to_rivers(
    da=glofas_monthly,
    rivers_gdf=rivers_only,
    variable_name="dis24",
)

historical_monthly_masked = mask_cmip_using_glofas(
    cmip_da=historical_monthly_interp,
    glofas_da=glofas_monthly_masked,
    threshold=RIVER_DISCHARGE_THRESHOLD,
)

scenario_monthly_masked = mask_cmip_using_glofas(
    cmip_da=scenario_monthly_interp,
    glofas_da=glofas_monthly_masked,
    threshold=RIVER_DISCHARGE_THRESHOLD,
)

print("Masked historical CMIP6:")
print(historical_monthly_masked)
print("\nMasked scenario CMIP6:")
print(scenario_monthly_masked)

Masked historical CMIP6:
<xarray.DataArray 'rivo' (month: 12, latitude: 130, longitude: 330)> Size: 2MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, 

## Step 8. Compute monthly CMIP6 percentage change

For each month, the notebook computes:

`percentage change = (scenario - historical) / historical * 100`

This percentage change is later applied to the present integrated GLOFAS basin statistics.

In [13]:
percent_change_monthly = compute_percent_change(
    historical_da=historical_monthly_masked,
    scenario_da=scenario_monthly_masked,
)

print(percent_change_monthly)

<xarray.Dataset> Size: 2MB
Dimensions:              (latitude: 130, longitude: 330, month: 12)
Coordinates:
  * latitude             (latitude) float32 520B 1.475 1.425 ... -4.925 -4.975
  * longitude            (longitude) float32 1kB -91.67 -91.62 ... -75.27 -75.23
  * month                (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
    spatial_ref          int64 8B 0
Data variables:
    rivo_percent_change  (month, latitude, longitude) float32 2MB nan ... nan


## Step 9. Apply CMIP6 deltas to the integrated GLOFAS basin statistics

For each month, this step aggregates the CMIP6 percentage change over level 12 basins, reads the integrated present-day GLOFAS basin statistics, applies the percentage change, saves level 12 output as CSV, and aggregates the result to level 5.

In [16]:
affine = get_affine_from_reference_file(GLOFAS_FILE)

for month in range(1, 2):
    print("\n" + "=" * 80)
    print(f"Processing month {month}")

    percent_change_l12 = aggregate_percent_change_on_basins(
        percent_change_ds=percent_change_monthly,
        country_basins=country_basins_l12,
        month=month,
        affine=affine,
    )

    present_integrated_l12 = read_integrated_l12_basin_statistics(month)

    scenario_l12 = apply_percent_delta_to_reanalysis(
        percent_change_l12=percent_change_l12,
        present_integrated_l12=present_integrated_l12,
    )

    scenario_l5 = aggregate_l12_to_l5(
        gdf_l12=scenario_l12,
        country_basins_l5=country_basins_l5,
    )

    l12_csv = OUTPUT_DIR / f"river_discharge_l12_{COUNTRY_FOLDER}_m{month}_{SCENARIO_EXPERIMENT}_{SCENARIO_START}_{SCENARIO_END}.csv"
    l5_csv = OUTPUT_DIR / f"river_discharge_l5_{COUNTRY_FOLDER}_m{month}_{SCENARIO_EXPERIMENT}_{SCENARIO_START}_{SCENARIO_END}.csv"
    l5_geojson = OUTPUT_DIR / f"river_discharge_l5_{COUNTRY_FOLDER}_m{month}_{SCENARIO_EXPERIMENT}_{SCENARIO_START}_{SCENARIO_END}.geojson"

    save_geodataframe_to_csv(scenario_l12, l12_csv)
    save_geodataframe_to_csv(scenario_l5, l5_csv)
    save_geodataframe_to_geojson(scenario_l5, l5_geojson)

    print("Saved level 12 CSV:", l12_csv)
    print("Saved level 5 CSV:", l5_csv)
    print("Saved level 5 GeoJSON:", l5_geojson)


Processing month 1
Saved level 12 CSV: ../data/atlas_data/ecuador/CNRM-ESM2-1/ssp370/river_discharge_l12_ecuador_m1_ssp370_2020_2050.csv
Saved level 5 CSV: ../data/atlas_data/ecuador/CNRM-ESM2-1/ssp370/river_discharge_l5_ecuador_m1_ssp370_2020_2050.csv
Saved level 5 GeoJSON: ../data/atlas_data/ecuador/CNRM-ESM2-1/ssp370/river_discharge_l5_ecuador_m1_ssp370_2020_2050.geojson


## Step 10. Check the outputs

This final step lists the files created in the output folder.

In [17]:
output_files = sorted(OUTPUT_DIR.glob("*"))

print(f"Number of output files: {len(output_files)}")
for output_file in output_files:
    print(output_file)

Number of output files: 3
../data/atlas_data/ecuador/CNRM-ESM2-1/ssp370/river_discharge_l12_ecuador_m1_ssp370_2020_2050.csv
../data/atlas_data/ecuador/CNRM-ESM2-1/ssp370/river_discharge_l5_ecuador_m1_ssp370_2020_2050.csv
../data/atlas_data/ecuador/CNRM-ESM2-1/ssp370/river_discharge_l5_ecuador_m1_ssp370_2020_2050.geojson
